## Ghouls, Goblins, and Ghosts... Boo!

This project classifies monsters (ghoul, goblin, or ghost) using simple physical measurements. It is a small, fun multi-class classification problem, good for a quick full pipeline (load, explore, train, predict, submit).

## Approach
1. Load the data and explore the features by monster type
2. Encode the categorical color feature
3. Train a baseline classifier and evaluate it with a validation split
4. Generate predictions and create the submission file
5. Submit to Kaggle and record the score

In [1]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token (starts with KGAT_) and press Enter: ")

!pip install -q -U kaggle
!kaggle competitions download -c ghouls-goblins-and-ghosts-boo
!unzip -oq ghouls-goblins-and-ghosts-boo.zip

Paste your Kaggle API token (starts with KGAT_) and press Enter: ··········
ghouls-goblins-and-ghosts-boo.zip: Skipping, found more recently modified local copy (use --force to force download)


In [2]:
!unzip -oq train.csv.zip
!unzip -oq test.csv.zip
!unzip -oq sample_submission.csv.zip
!ls

ghouls-goblins-and-ghosts-boo.zip  sample_submission.csv.zip  train.csv
sample_data			   test.csv		      train.csv.zip
sample_submission.csv		   test.csv.zip


In [3]:
import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [4]:
train.sample(1)

,id,bone_length,rotting_flesh,hair_length,has_soul,color,type
46,93,0.481716,0.527251,0.74027,0.700857,blood,Ghoul


In [5]:
test.sample(1)

,id,bone_length,rotting_flesh,hair_length,has_soul,color
366,623,0.480217,0.694797,0.717046,0.623958,clear


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 371 entries, 0 to 370
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             371 non-null    int64  
 1   bone_length    371 non-null    float64
 2   rotting_flesh  371 non-null    float64
 3   hair_length    371 non-null    float64
 4   has_soul       371 non-null    float64
 5   color          371 non-null    object 
 6   type           371 non-null    object 
dtypes: float64(4), int64(1), object(2)
memory usage: 20.4+ KB


In [7]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 529 entries, 0 to 528
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             529 non-null    int64  
 1   bone_length    529 non-null    float64
 2   rotting_flesh  529 non-null    float64
 3   hair_length    529 non-null    float64
 4   has_soul       529 non-null    float64
 5   color          529 non-null    object 
dtypes: float64(4), int64(1), object(1)
memory usage: 24.9+ KB


In [8]:
train.shape

(371, 7)

In [9]:
test.shape

(529, 6)

In [10]:
train["type"].value_counts()

,count
type,
Ghoul,129
Goblin,125
Ghost,117


In [11]:
train["color"].value_counts()

,count
color,
white,137
clear,120
green,42
black,41
blue,19
blood,12


In [12]:
train = pd.get_dummies(train, columns=["color"])
test = pd.get_dummies(test, columns=["color"])

In [13]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 371 entries, 0 to 370
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             371 non-null    int64  
 1   bone_length    371 non-null    float64
 2   rotting_flesh  371 non-null    float64
 3   hair_length    371 non-null    float64
 4   has_soul       371 non-null    float64
 5   type           371 non-null    object 
 6   color_black    371 non-null    bool   
 7   color_blood    371 non-null    bool   
 8   color_blue     371 non-null    bool   
 9   color_clear    371 non-null    bool   
 10  color_green    371 non-null    bool   
 11  color_white    371 non-null    bool   
dtypes: bool(6), float64(4), int64(1), object(1)
memory usage: 19.7+ KB


In [14]:
x = train[["bone_length", "rotting_flesh", "hair_length", "has_soul", "color_black", "color_blood", "color_blue", "color_clear", "color_green", "color_white"]]
y = train["type"]

In [15]:
test[["bone_length", "rotting_flesh", "hair_length", "has_soul", "color_black", "color_blood", "color_blue", "color_clear", "color_green", "color_white"]]

,bone_length,rotting_flesh,hair_length,has_soul,color_black,color_blood,color_blue,color_clear,color_green,color_white
0,0.471774,0.387937,0.706087,0.698537,True,False,False,False,False,False
1,0.427332,0.645024,0.565558,0.451462,False,False,False,False,False,True
2,0.549602,0.491931,0.660387,0.449809,True,False,False,False,False,False
3,0.638095,0.682867,0.471409,0.356924,False,False,False,False,False,True
4,0.361762,0.583997,0.377256,0.276364,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
524,0.377573,0.390158,0.696465,0.355373,False,False,True,False,False,False
525,0.229161,0.601265,0.191282,0.475115,False,False,False,True,False,False
526,0.510497,0.498347,0.708020,0.714154,False,False,False,False,False,True
527,0.331472,0.765835,0.338207,0.193431,False,False,False,True,False,False


In [16]:
x.columns.tolist()

['bone_length',
 'rotting_flesh',
 'hair_length',
 'has_soul',
 'color_black',
 'color_blood',
 'color_blue',
 'color_clear',
 'color_green',
 'color_white']

In [17]:
x.shape, y.shape

((371, 10), (371,))

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(x_train, y_train)
print("Validation accuracy:", round(accuracy_score(y_val, model.predict(x_val)), 4))

Validation accuracy: 0.7733


In [19]:
model.fit(x, y)

x_test = test[["bone_length", "rotting_flesh", "hair_length", "has_soul", "color_black", "color_blood", "color_blue", "color_clear", "color_green", "color_white"]]
pred = model.predict(x_test)

pd.DataFrame({"id": test["id"], "type": pred}).to_csv("submission.csv", index=False)

In [20]:
!kaggle competitions submit -c ghouls-goblins-and-ghosts-boo -f submission.csv -m "RandomForest baseline"

100% 5.27k/5.27k [00:00<00:00, 8.49kB/s]
99 submissions remaining today.
Successfully submitted to Ghouls, Goblins, and Ghosts... Boo! 

In [21]:
import pickle

pickle.dump(model, open("ghosts_model.pkl", "wb"))